In [1]:
import os

os.chdir('../')

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [3]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir,'Chest-CT-Scan-data')
        create_directories([Path(training.root_dir)])
        
        training_config = TrainingConfig(
            root_dir = Path(training.root_dir),
            trained_model_path = Path(training.trained_model_path),
            updated_base_model_path = Path(prepare_base_model.updated_base_model_path),
            training_data = Path(training_data),
            params_epochs = params.EPOCHS,
            params_batch_size = params.BATCH_SIZE,
            params_is_augmentation = params.AUGMENTATION,
            params_image_size = params.IMAGE_SIZE
        )
        return training_config

In [5]:
import os
import tensorflow as tf
import time

In [6]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(self.config.updated_base_model_path)
    
    def train_valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.2
        )
        
        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = 'bilinear',
        )
        
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = 'validation',
            shuffle = False,
            **dataflow_kwargs            
        )
        
        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range = 40,
                horizontal_flip = True,
                width_shift_range = 0.2,
                height_shift_range = 0.2,
                shear_range = 0.2,
                zoom_range = 0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator
            
        self.train_generator = train_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = 'training',
            shuffle = True,
            **dataflow_kwargs
        )
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
        
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size
        
        self.model.fit(
            self.train_generator,
            epochs = self.config.params_epochs,
            steps_per_epoch = self.steps_per_epoch,
            validation_steps = self.validation_steps,
            validation_data = self.valid_generator
        )
        
        self.save_model(
            path = self.config.trained_model_path,
            model = self.model
        )

In [8]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e

[2026-03-08 21:10:29,111: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-08 21:10:29,113: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-08 21:10:29,114: INFO: common: created directory at: artifacts]
[2026-03-08 21:10:29,114: INFO: common: created directory at: artifacts/training]
Found 68 images belonging to 2 classes.
Found 275 images belonging to 2 classes.
Epoch 1/10


2026-03-08 21:10:29.431931: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


17/17 [==============================] - 4s 173ms/step - loss: 17.6215 - accuracy: 0.5019 - val_loss: 17.8721 - val_accuracy: 0.6094
Epoch 2/10
17/17 [==============================] - 3s 165ms/step - loss: 16.4869 - accuracy: 0.5019 - val_loss: 8.6816 - val_accuracy: 0.3906
Epoch 3/10
17/17 [==============================] - 3s 154ms/step - loss: 6.3050 - accuracy: 0.6564 - val_loss: 10.7880 - val_accuracy: 0.6094
Epoch 4/10
17/17 [==============================] - 3s 157ms/step - loss: 9.6170 - accuracy: 0.5830 - val_loss: 17.5156 - val_accuracy: 0.6094
Epoch 5/10
17/17 [==============================] - 3s 165ms/step - loss: 2.9093 - accuracy: 0.7992 - val_loss: 0.1972 - val_accuracy: 0.9688
Epoch 6/10
17/17 [==============================] - 3s 160ms/step - loss: 2.7327 - accuracy: 0.8340 - val_loss: 0.0140 - val_accuracy: 1.0000
Epoch 7/10
17/17 [==============================] - 3s 166ms/step - loss: 3.3182 - accuracy: 0.7645 - val_loss: 0.4861 - val_accuracy: 0.8750
Epoch 8/10
1